In [50]:
import sys
!{sys.executable} -m pip install pymilvus
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)

I0601 13:44:07.615423  487861 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(78, generation: 1)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [51]:
print("Collections:", milvus_client.list_collections())

Collections: []


In [52]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [53]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 466651096139950660, 'consistency_level': 2, 'consistency_level_name': 'Bounded', 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False, 'created_timestamp': 466698699794022417, 'update_timestamp': 466698699794022417}


In [54]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [55]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [56]:
# prepare data
!{sys.executable} -m pip install pymupdf

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

I0601 13:44:22.938991  488159 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(78, generation: 1)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [57]:
# vectorize data
!{sys.executable} -m pip install sentence-transformers torch
!{sys.executable} -m pip install tf_keras

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

I0601 13:44:27.930440  488220 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(78, generation: 1)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


I0601 13:44:29.068400  488252 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(78, generation: 1)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [58]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [59]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [60]:
print(result)

data: [[{'id': 466651096138393397, 'distance': 29.125185012817383, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mog

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = (
    "FILL_API_KEY_HERE"
)

In [ ]:
!pip install google-genai

import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text 
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade google-genai


In [68]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Jesteś ekspertem ds. sztucznej inteligencji. Przeczytaj poniższy kontekst i odpowiedz na pytanie użytkownika.
    Wykorzystaj informacje zawarte w kontekście, aby ustrukturyzować swoją wypowiedź.
    
    Kontekst:
    {context}
    
    Pytanie:
    {query}
    
    Odpowiedź:"""
    return prompt


def get_embedding(text: str) -> list[float]:
    response = gemini_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config={"output_dimensionality": VECTOR_LENGTH}
    )
    return response.embeddings[0].values



def rag(model, query: str) -> str:
    # having all prepared functions, you can combine them together and try to build your own RAG!

    try:
        query_vector = get_embedding(query)

        search_results = milvus_client.search(
            collection_name="rag_texts_and_embeddings",
            data=[query_vector],
            limit=3,
            output_fields=["text"],
        )

        context_chunks = []
        for hits in search_results:
            for hit in hits:
                context_chunks.append(hit.get("entity").get("text"))

        full_context = "\n\n".join(context_chunks)

        final_prompt = build_prompt(context=full_context, query=query)

        response = gemini_client.models.generate_content(
            model=model,
            contents=final_prompt,
        )
        return response.text

    except Exception as e:
        return f"Wystąpił błąd w systemie RAG: {e}"


In [72]:
test_queries = [
    "Czym jest sztuczna inteligencja?",
    "Co to są wektory?",
]

MODEL_NAME = "gemini-2.5-flash"


for i, query in enumerate(test_queries, 1):
    print(f"Test {i}: Pytanie: '{query}'")
    print("-" * 40)
    
    response = rag(model=MODEL_NAME, query=query)
    
    print(f"RAG response:\n{response}")
    print("\n" + "="*60 + "\n")


Test 1: Pytanie: 'Czym jest sztuczna inteligencja?'
----------------------------------------
RAG response:
Jako ekspert ds. sztucznej inteligencji, na podstawie dostarczonego kontekstu, mogę przedstawić kompleksowe spojrzenie na to, czym jest AI, jakie ma zastosowania, oraz jakie wyzwania i implikacje za sobą niesie.

### Definicja i Charakterystyka Sztucznej Inteligencji (AI)

Sztuczna inteligencja, w ujęciu przedstawionym w tekście, to przede wszystkim **technologia bazująca na algorytmach**, która ma potencjał do:
*   **Wspierania i automatyzacji:** Umożliwia masowy wzrost produktywności oraz automatyzację procesów.
*   **Modelowania i przewidywania:** Wykorzystywana jest do analizy danych, ich modelowania i dokonywania prognoz.
*   **Wykonywania złożonych zadań:** Jest zdolna do wykonywania zadań, które dotychczas wymagały ludzkiej inteligencji, takich jak pisanie esejów czy personalizacja ścieżek edukacyjnych.

Tekst sugeruje, że AI to **zbiór narzędzi i rozwiązań**, które wymagaj